# Agent 2 — Data Ingestion

**What this notebook does:**  
1. Loads our 170-company universe from the STOXX 600 outperformers Excel file  
2. Filters the four professor CSV files to only those companies  
3. Downloads stock prices from Yahoo Finance  
4. Merges everything into one master dataset  

**How to present this to investors:**  
> *Our universe is the 170 STOXX Europe 600 companies that outperformed the index on both a 5-year and 10-year total return basis. We then filter these through four Bloomberg/professor-provided ESG datasets and apply sector exclusions, producing a clean master dataset for our 13-agent pipeline.*

In [1]:
import pandas as pd
import numpy as np
import os
import re
import openpyxl
import yfinance as yf
from datetime import date

TODAY = str(date.today())
DATA_DIR   = "../data/provided"
MARKET_DIR = "../data/market"
STOXX_FILE = "../STOXX600_Outperformers_5Y_10Y.xlsx"
os.makedirs(MARKET_DIR, exist_ok=True)
print(f"Run date: {TODAY}")

Run date: 2026-05-12


## Step 1 — Build the 170-company universe from the STOXX 600 outperformers file

In [2]:
# Load the pre-matched universe reference file (168 of 170 companies matched to Bloomberg IDs)
df_universe = pd.read_csv("../data/provided/universe_170.csv")

print(f"Universe loaded: {len(df_universe)} companies")
print(f"Matched to Bloomberg ID: {df_universe['idBbCompany'].notna().sum()}/170")
print(f"Yahoo Finance tickers (from Bloomberg source): {df_universe['yf_ticker'].notna().sum()}/170")
print()
df_universe[['rank', 'company', 'return_10y_pct', 'bb_ticker', 'yf_ticker']].head(10)

Universe loaded: 170 companies
Matched to Bloomberg ID: 168/170
Yahoo Finance tickers (from Bloomberg source): 170/170



,rank,company,return_10y_pct,bb_ticker,yf_ticker
0,1,argenx SE,6000.5,ARGX BB,ARGX.BR
1,2,Games Workshop Group PLC,4156.2,GAW LN,GAW.L
2,3,ASM International N.V.,2693.5,ASM NA,ASM.AS
3,4,BE Semiconductor Industries N.V.,2220.7,BESI NA,BESI.AS
4,5,Swissquote Group Holding Ltd.,1985.9,SQN SW,SQN.SW
5,6,ASML Holding NV,1550.8,ASML NA,ASML.AS
6,7,Zegona Communications Plc,1353.7,ZEG LN,ZEG.L
7,8,VAT Group AG,1353.3,VACN SW,VACN.SW
8,9,AIXTRON SE,1077.1,AIXA GR,AIXA.DE
9,10,Sectra AB Class B,1024.7,SECTB SS,SECTB.ST


## Step 2 — Load equityBicsV2 and filter to our universe

In [3]:
JOIN_KEY = "idBbCompany"

# Get the set of Bloomberg company IDs for our 170 companies
target_ids = set(df_universe['idBbCompany'].dropna().astype(int).tolist())
print(f"Target universe: {len(target_ids)} unique Bloomberg company IDs")

print("Loading equityBicsV2.csv (takes ~15 seconds)...")
df_ids = pd.read_csv(f"{DATA_DIR}/equityBicsV2.csv", low_memory=False)
print(f"Loaded: {len(df_ids):,} rows | {df_ids[JOIN_KEY].nunique():,} unique companies")

Target universe: 168 unique Bloomberg company IDs
Loading equityBicsV2.csv (takes ~15 seconds)...


Loaded: 197,562 rows | 67,543 unique companies


In [4]:
# Filter equityBicsV2 to our universe and keep ONE row per company (primary equity)
df_filtered = df_ids[df_ids[JOIN_KEY].isin(target_ids)].copy()
print(f"Rows for our universe before exchange filter: {len(df_filtered):,}")

# Keep only European exchange listings — Bloomberg idBbCompany is not globally unique;
# the same numeric ID can appear for a European and an Asian company in the professor's export.
EUROPEAN_EXCH = {
    'GR','LN','FP','IM','SS','AV','BB','PW','SM','NO','FH','DC',
    'PO','BQ','SW','MM','AR','HB','RO','RM','CP','TE','T1','S1',
    'B3','L1','GZ','IX','QX','TQ','EB','QE','S4','PZ','L3','LI',
    'I2','BU','TH','QT','ID',   # ID = Euronext Dublin
}
if 'exchCode' in df_filtered.columns:
    non_eu = df_filtered[~df_filtered['exchCode'].isin(EUROPEAN_EXCH)]
    if len(non_eu) > 0:
        print(f"Dropping {len(non_eu)} non-European rows (exchCode: {non_eu['exchCode'].unique().tolist()}):")
        print(non_eu[['idBbGlobalCompanyName', 'exchCode']].drop_duplicates().head(10).to_string(index=False))
    df_filtered = df_filtered[df_filtered['exchCode'].isin(EUROPEAN_EXCH)]
    print(f"Rows after European exchange filter: {len(df_filtered):,}")

# Prefer common equity rows where possible
if 'securityTyp' in df_filtered.columns:
    common = df_filtered[df_filtered['securityTyp'].str.contains('Common|EQS', case=False, na=False)]
    df_company = common.drop_duplicates(subset=JOIN_KEY, keep='first') if len(common) > 0 else df_filtered.drop_duplicates(subset=JOIN_KEY, keep='first')
else:
    df_company = df_filtered.drop_duplicates(subset=JOIN_KEY, keep='first')

# Attach universe metadata (rank, returns, Yahoo Finance ticker)
universe_meta = df_universe[[JOIN_KEY, 'rank', 'company', 'return_5y_pct', 'return_10y_pct', 'bb_ticker', 'yf_ticker']].dropna(subset=[JOIN_KEY]).copy()
universe_meta[JOIN_KEY] = universe_meta[JOIN_KEY].astype(int)
df_company = df_company.merge(universe_meta, on=JOIN_KEY, how='left')

print(f"Companies in filtered dataset: {len(df_company)}")
df_company[['idBbGlobalCompanyName', 'rank', 'return_10y_pct', 'bb_ticker', 'yf_ticker']].sort_values('rank').head(10)

Rows for our universe before exchange filter: 4,766
Dropping 336 non-European rows (exchCode: ['NZ', 'HK', 'SP', 'H1', 'H2', 'PL', 'LX', nan, 'GA', 'SJ', 'LR', 'LH', 'UZ', 'US', 'BZ', 'BH', 'PE']):
 idBbGlobalCompanyName exchCode
    Credit Agricole SA       NZ
     HSBC Holdings PLC       HK
Standard Chartered PLC       HK
     HSBC Holdings PLC       SP
     HSBC Holdings PLC       H1
Standard Chartered PLC       H1
     HSBC Holdings PLC       H2
Standard Chartered PLC       H2
           Euronext NV       PL
      ArcelorMittal SA       LX
Rows after European exchange filter: 4,430
Companies in filtered dataset: 168


,idBbGlobalCompanyName,rank,return_10y_pct,bb_ticker,yf_ticker
142,Argenx SE,1,6000.5,ARGX BB,ARGX.BR
135,Games Workshop Group PLC,2,4156.2,GAW LN,GAW.L
46,ASM International NV,3,2693.5,ASM NA,ASM.AS
28,BE Semiconductor Industries NV,4,2220.7,BESI NA,BESI.AS
40,Swissquote Group Holding SA,5,1985.9,SQN SW,SQN.SW
29,ASML Holding NV,6,1550.8,ASML NA,ASML.AS
145,Zegona Communications plc,7,1353.7,ZEG LN,ZEG.L
150,VAT Group AG,8,1353.3,VACN SW,VACN.SW
16,AIXTRON SE,9,1077.1,AIXA GR,AIXA.DE
162,Sectra AB,10,1024.7,SECTB SS,SECTB.ST


## Step 3 — Apply sector exclusions (gambling and defense)

In [5]:
# Exclusion keywords in BICS classification columns
# Strip "(excl ...)" parentheticals first so "Hotel & Motel (excl Casino Hotel)"
# does NOT trigger the casino keyword.
EXCLUDE_PATTERNS = [
    r'\bcasinos?\b',
    r'\bgaming\b',
    r'\bgambling\b',
    r'\blottery\b',
    r'\bdefense\b',
    r'\bdefence\b',
    r'\bweapons?\b',
    r'\bammunition\b',
    r'\btobacco\b',
]

class_cols = [c for c in df_company.columns if 'classificationLevelName' in c]
print(f"BICS classification columns: {class_cols}")

def is_excluded(row):
    for col in class_cols:
        raw = str(row.get(col, ''))
        val = re.sub(r'\(excl[^)]*\)', '', raw)  # remove "(excl ...)" so we don't hit false positives
        for pat in EXCLUDE_PATTERNS:
            if re.search(pat, val, re.IGNORECASE):
                return True
    return False

df_company['excluded'] = df_company.apply(is_excluded, axis=1)
excluded = df_company[df_company['excluded']]
df_company = df_company[~df_company['excluded']].copy()

print(f"Excluded companies ({len(excluded)}):")
for _, row in excluded.iterrows():
    reason = next(
        (str(row[c]) for c in class_cols
         if any(re.search(p,
                          re.sub(r'\(excl[^)]*\)', '', str(row[c])),
                          re.IGNORECASE)
                for p in EXCLUDE_PATTERNS)),
        'unknown'
    )
    print(f"  - {row.get('idBbGlobalCompanyName', 'N/A')} | Reason: {reason}")

print(f"\nUniverse after exclusions: {len(df_company)} companies")

BICS classification columns: ['classificationLevelName1', 'classificationLevelName2', 'classificationLevelName3', 'classificationLevelName4', 'classificationLevelName5', 'classificationLevelName6', 'classificationLevelName7']
Excluded companies (1):
  - Safran SA | Reason: Aerospace & Defense

Universe after exclusions: 167 companies


## Step 4 — Load and merge ESG environmental + social data

In [6]:
print("Loading ESG environmental + social data...")
df_esg = pd.read_csv(f"{DATA_DIR}/esgEnvironmentalSocialConsolidatedV4.csv", low_memory=False)
print(f"Loaded: {len(df_esg):,} rows x {len(df_esg.columns)} columns")

# Filter to our universe
df_esg = df_esg[df_esg[JOIN_KEY].isin(target_ids)]
print(f"After universe filter: {len(df_esg)} rows")

# Keep most recent reporting period per company
if 'latestPeriodEndCsr' in df_esg.columns and df_esg[JOIN_KEY].duplicated().any():
    df_esg = df_esg.sort_values('latestPeriodEndCsr', ascending=False).drop_duplicates(JOIN_KEY, keep='first')
    print(f"After keeping most recent year per company: {len(df_esg)} rows")

print(f"ESG rows matched to universe: {len(df_esg)}")

Loading ESG environmental + social data...


Loaded: 28,186 rows x 463 columns
After universe filter: 324 rows
After keeping most recent year per company: 163 rows
ESG rows matched to universe: 163


## Step 5 — Load and merge governance data

In [7]:
print("Loading governance data...")
df_gov = pd.read_csv(f"{DATA_DIR}/esgGovernanceConsolidatedV4.csv", low_memory=False)
print(f"Loaded: {len(df_gov):,} rows x {len(df_gov.columns)} columns")

df_gov = df_gov[df_gov[JOIN_KEY].isin(target_ids)]

if 'latestPeriodEndCsr' in df_gov.columns and df_gov[JOIN_KEY].duplicated().any():
    df_gov = df_gov.sort_values('latestPeriodEndCsr', ascending=False).drop_duplicates(JOIN_KEY, keep='first')

print(f"Governance rows matched to universe: {len(df_gov)}")

Loading governance data...


Loaded: 28,186 rows x 148 columns
Governance rows matched to universe: 163


## Step 6 — Load and merge EU Taxonomy data

In [8]:
print("Loading EU Taxonomy data...")
df_tax = pd.read_csv(f"{DATA_DIR}/legalEntityEuTaxonomy.csv", low_memory=False)
print(f"Loaded: {len(df_tax):,} rows x {len(df_tax.columns)} columns")

df_tax = df_tax[df_tax[JOIN_KEY].isin(target_ids)]

if 'latestPeriodEndCsr' in df_tax.columns and df_tax[JOIN_KEY].duplicated().any():
    df_tax = df_tax.sort_values('latestPeriodEndCsr', ascending=False).drop_duplicates(JOIN_KEY, keep='first')

print(f"Taxonomy rows matched to universe: {len(df_tax)}")

Loading EU Taxonomy data...
Loaded: 48,555 rows x 53 columns
Taxonomy rows matched to universe: 168


## Step 7 — Merge into one master table

In [9]:
master = df_company.copy()

# Drop duplicate identifier columns from ESG/gov/tax before merging
drop_cols = ['rc', 'idBbGlobal', 'idBbGlobalCompany', 'idBbGlobalCompanyName',
             'primSecurityCompIdBbGlobal', 'eqyFundCrncy', 'cntryOfIncorporation',
             'cntryOfDomicile', 'eqyConsolidated']

for df_src, label in [(df_esg, '_esg'), (df_gov, '_gov'), (df_tax, '_tax')]:
    cols_to_drop = [c for c in drop_cols if c in df_src.columns and c != JOIN_KEY]
    df_src_clean = df_src.drop(columns=cols_to_drop, errors='ignore')
    master = master.merge(df_src_clean, on=JOIN_KEY, how='left', suffixes=('', label))
    print(f"After merging {label}: {master.shape}")

master['data_vintage'] = TODAY
print(f"\nFinal master table: {len(master)} companies x {len(master.columns)} columns")
master[['idBbGlobalCompanyName', 'rank', 'return_10y_pct', 'yf_ticker']].sort_values('rank').head(15)

After merging _esg: (167, 490)
After merging _gov: (167, 629)
After merging _tax: (167, 676)

Final master table: 167 companies x 677 columns


,idBbGlobalCompanyName,rank,return_10y_pct,yf_ticker
141,Argenx SE,1,6000.5,ARGX.BR
134,Games Workshop Group PLC,2,4156.2,GAW.L
46,ASM International NV,3,2693.5,ASM.AS
28,BE Semiconductor Industries NV,4,2220.7,BESI.AS
40,Swissquote Group Holding SA,5,1985.9,SQN.SW
29,ASML Holding NV,6,1550.8,ASML.AS
144,Zegona Communications plc,7,1353.7,ZEG.L
149,VAT Group AG,8,1353.3,VACN.SW
16,AIXTRON SE,9,1077.1,AIXA.DE
161,Sectra AB,10,1024.7,SECTB.ST


## Step 8 — Download stock prices from Yahoo Finance

In [10]:
tickers = master['yf_ticker'].dropna().unique().tolist()
print(f"Downloading prices for {len(tickers)} tickers...")
print("Sample:", tickers[:8])

price_file = f"{MARKET_DIR}/prices_{TODAY}.csv"

if os.path.exists(price_file):
    print(f"Loading cached prices from {price_file}")
    prices = pd.read_csv(price_file, index_col=0, parse_dates=True)
else:
    print("Downloading from Yahoo Finance (takes 2–4 minutes)...")
    raw = yf.download(tickers, start="2015-01-01", auto_adjust=True, progress=True)
    prices = raw["Close"]
    prices.to_csv(price_file)
    print(f"Saved to {price_file}")

downloaded = set(prices.columns.tolist())
missing_prices = [t for t in tickers if t not in downloaded]
print(f"\nPrice data: {prices.shape[0]} trading days x {prices.shape[1]} stocks")
print(f"Missing prices ({len(missing_prices)}): {missing_prices[:10]}")

Sample: ['SEBA.ST', 'ALV.DE', 'RXL.PA', 'ISP.MI', 'RWE.DE', 'NDA.DE', 'CBK.DE', 'GBF.DE']


$SEBA.ST: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-05-12)


[                       0%                       ]

[                       1%                       ]  2 of 167 completed

[*                      2%                       ]  3 of 167 completed

...

[*********************100%***********************]  167 of 167 completed



28 Failed downloads:


['SEBA.ST', 'LAGRB.ST', 'TEL2B.ST', 'SSABB.ST', 'TRELB.ST', 'SECTB.ST', 'DIE.BR', 'VACN.SW', 'ALKB.CO', 'SCYR.MC', 'ELI.BR', 'BAMI.MI', 'PKN.WA', 'UCB.BR', 'STAN.L', 'KBC.BR', 'ISP.MI', 'BESI.AS', 'OMV.VI', 'SAN.MC', 'INVEB.ST', 'ADDTB.ST', 'VOLVB.ST', 'UNI.MI']: possibly delisted; no price data found  (1d 2015-01-01 -> 2026-05-12)


['NDAFI.HE', 'STM.MI', 'SYDB.CO', 'PST.MI']: possibly delisted; no timezone found


Saved to ../data/market/prices_2026-05-12.csv

Price data: 2934 trading days x 167 stocks
Missing prices (0): []


## Step 9 — Data quality check

In [11]:
missing = master.isnull().mean().mul(100).round(1)
missing_summary = missing[missing > 0].sort_values(ascending=False)

print(f"Companies in master: {len(master)}")
print(f"Total columns: {len(master.columns)}")
print(f"Columns with any missing data: {len(missing_summary)}")
print("\nTop 20 most incomplete columns:")
print(missing_summary.head(20).to_string())

Companies in master: 167
Total columns: 677
Columns with any missing data: 644

Top 20 most incomplete columns:
noxIntensPerRevPsngrMiKms               100.0
cadmiumEmissions                        100.0
purchasedElecOthRenewables              100.0
co2IntensityPerGwhSold                  100.0
mangnseEmissns                          100.0
thirdPartyCertfidForestlnd              100.0
pctOfFloorAreaCovrdByEnrgyCertficatn    100.0
pctPortfolioEnergyRtg                   100.0
vehicleIncidentRtWorkforce              100.0
vehIncidentRateCntrctr                  100.0
vehicleIncidentRtEmployees              100.0
bbbeeAndBlackHdsaOwnershipPct           100.0
bbbeeOverallScore                       100.0
bbbeeRatingLevel                        100.0
enrgyIntensRevPsngrMiKms                100.0
powerPurchaseAgreement                  100.0
ghgIntensityPerRpm                      100.0
purchasedElectricityCoal                100.0
electGenProdThermMultiPct               100.0
purchasedElect

## Step 10 — Save the master dataset

In [12]:
os.makedirs("../outputs/scores", exist_ok=True)
output_path = f"../outputs/scores/master_dataset_{TODAY}.csv"
master.to_csv(output_path, index=False)
print(f"Master dataset saved: {output_path}")
print(f"Shape: {master.shape[0]} companies x {master.shape[1]} columns")
print(f"Data vintage: {TODAY}")
print(f"\nCompanies by rank (top 20):")
print(master[['idBbGlobalCompanyName', 'rank', 'return_10y_pct', 'yf_ticker']]
      .sort_values('rank').head(20).to_string(index=False))

Master dataset saved: ../outputs/scores/master_dataset_2026-05-12.csv
Shape: 167 companies x 677 columns
Data vintage: 2026-05-12

Companies by rank (top 20):
         idBbGlobalCompanyName  rank  return_10y_pct yf_ticker
                     Argenx SE     1          6000.5   ARGX.BR
      Games Workshop Group PLC     2          4156.2     GAW.L
          ASM International NV     3          2693.5    ASM.AS
BE Semiconductor Industries NV     4          2220.7   BESI.AS
   Swissquote Group Holding SA     5          1985.9    SQN.SW
               ASML Holding NV     6          1550.8   ASML.AS
     Zegona Communications plc     7          1353.7     ZEG.L
                  VAT Group AG     8          1353.3   VACN.SW
                    AIXTRON SE     9          1077.1   AIXA.DE
                     Sectra AB    10          1024.7  SECTB.ST
         STMicroelectronics NV    11           995.1    STM.MI
                 CD Projekt SA    12           991.3    CDR.WA
                    Ad

## ✅ Notebook complete

You now have:
- A **master dataset** with all four professor CSV files merged, filtered to the 170 STOXX outperformers
- **Sector exclusions** applied (gambling, defense, tobacco)
- **5 years of stock prices** cached locally
- A **data quality report** showing which fields have missing values

**Next:** Open `agent03_data_quality.ipynb` to run the full data audit.